# Kuiper：按获批轨道参数补全数量缺口

官方批准的是壳层参数，不是未来逐星 TLE。从完整真实目录估算数量缺口，生成 synthetic 模拟位置；缺口不是已核实的未发射名单。升轨卫星和备份星也计入已有数量，观测点可见性不参与数量估计。

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


In [ ]:
from synthetic_orbits import approved_shells, custom_shells, build_virtual
CONSTELLATION = "kuiper"
MODE = "approved_missing"
SEED = 20261005
scenario, times = read_scenario(ROOT)
print("Common epoch:", times[0])
print("Mode:", MODE, "Seed:", SEED)


In [ ]:
shells = approved_shells(CONSTELLATION, ROOT)
display(pd.DataFrame(shells))


## 生成并保存轨道参数
官方字段与假设字段分别记录。固定 seed 可复现 RAAN 起点、相位及缺口模板抽样。使用圆形 SGP4 平均轨道和零阻力，不模拟编队控制、升轨或碰撞规避。RAAN/相位不是运营商实测值，也不是已知空轨位。

In [ ]:
real_catalog = load_snapshot(CONSTELLATION, root=ROOT)
virtual, orbit_meta = build_virtual(
    CONSTELLATION, MODE, scenario["start_time_utc"], records=real_catalog,
    seed=SEED, root=ROOT, max_age_days=scenario["max_element_age_days"],
)
print("Generated synthetic satellites:", len(virtual))
orbit_dir = ROOT / orbit_meta["orbit_directory"]
display(pd.read_csv(orbit_dir / "population.csv"))
print("Saved mean elements:", orbit_dir / "synthetic_mean_elements.csv")


## 用统一时间轴与 CSV 格式传播
真实、获批壳层模拟、自定义模拟分别保存。零缺口时输出有表头的空 CSV 和全零 counts，属于有效结果。

In [ ]:
result = export_tracks(
    virtual, CONSTELLATION, source_type=orbit_meta["assumptions"]["source_type"],
    root=ROOT, extra_metadata=orbit_meta,
)
display(preview_export(result))


## 将来合并
默认不执行。指定相同 scenario.json 和快照的真实 CSV 后合并。同一星座的 `approved_missing` 与 `custom` notebook属于不同模拟场景，不能同时计入同一星座。合并函数检查时间、观测点、筛选设置、输入哈希和重复 ID。

In [ ]:
REAL_CSV = None  # Set an explicit path to the matching real visibility CSV.
if REAL_CSV is not None:
    merged = merge_exports(
        [REAL_CSV, result["visible_csv"]],
        ROOT / "outputs" / "merged" / f"{CONSTELLATION}_{MODE}_{stamp()}.csv",
    )
    print(merged)
